# Spotify Songs’ Genre Segmentation & Recommendation System
**Complete End-to-End Machine Learning Project Model**

---

### Project Architecture
1. **Data Preprocessing & Feature Engineering:** Cleaning missing metadata, extracting release year, converting track duration to minutes, normalizing base titles, and deduplicating `track_id` & `(track_name, track_artist)` entries (`32,833` raw rows $\rightarrow$ `26,228` unique songs).
2. **Exploratory Data Analysis (EDA) & Correlation Matrix:** Analyzing genre/subgenre distributions, track popularity, auditory profiles across the 6 macro-genres (`edm`, `latin`, `pop`, `r&b`, `rap`, `rock`), temporal trends, and pairwise feature correlations.
3. **K-Means Clustering & Cluster Validation:** Standardizing 9 continuous audio features (`StandardScaler`), selecting optimal clusters ($k=6$) via Elbow Method (Inertia) and Silhouette Score, projecting clusters in 2D PCA space, and mapping cluster compositions against `playlist_genre` and `playlist_name`.
4. **Hybrid Recommendation Engine & Evaluation:** Combining coarse-grained K-Means cluster filtering with fine-grained Cosine Similarity ranking (supporting both **Song-to-Song** and **Mood/Audio-Slider** queries), evaluated via quantitative metrics.
5. **Model Serialization:** Exporting fitted `StandardScaler`, `KMeans`, feature matrix, and processed catalog (`spotify_recommender_artifacts.pkl`) for deployment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.dpi'] = 120

## 1. Data Preprocessing & Feature Engineering

In [ ]:
# Load dataset (update filename if running locally with spotify_songs.csv)
DATA_PATH = 'upload_29247127-d163-4eee-859b-97c45b290960.csv'
df = pd.read_csv(DATA_PATH)
print(f"Raw dataset shape: {df.shape}")

# Drop rows with missing track metadata
df_clean = df.dropna(subset=['track_name', 'track_artist', 'track_album_name']).copy()

# Feature engineering: extract release_year and duration_min
df_clean['release_year'] = pd.to_datetime(
    df_clean['track_album_release_date'], errors='coerce', format='mixed'
).dt.year.astype(int)
df_clean['duration_min'] = df_clean['duration_ms'] / 60000.0

# Create normalized base title to prevent recommending duplicate editions/remasters
df_clean['base_track_name'] = (
    df_clean['track_name']
    .str.replace(r'\s*[\-\(].*$', '', regex=True)
    .str.strip()
    .str.lower()
)

# Deduplicate unique songs (retaining highest popularity) & filter invalid zero-tempo
df_unique = (
    df_clean.sort_values('track_popularity', ascending=False)
    .drop_duplicates(subset=['track_id'])
    .drop_duplicates(subset=['track_name', 'track_artist'])
    .query('tempo > 0')
    .reset_index(drop=True)
)

print(f"Valid playlist entries: {len(df_clean):,}")
print(f"Unique deduplicated songs for modeling: {len(df_unique):,}")
df_unique.head(3)

## 2. Exploratory Data Analysis (EDA) & Correlation Matrix

In [ ]:
# 2.1 Genre Distribution, Top Subgenres, and Popularity by Genre
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

genre_counts = df_clean['playlist_genre'].value_counts()
sns.barplot(x=genre_counts.index, y=genre_counts.values, ax=axes[0], palette='viridis')
axes[0].set_title('Track Distribution by Playlist Genre', fontweight='bold')
axes[0].set_xlabel('Playlist Genre')
axes[0].set_ylabel('Number of Tracks')
for i, v in enumerate(genre_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontsize=9)

subgenre_counts = df_clean['playlist_subgenre'].value_counts().head(10)
sns.barplot(y=subgenre_counts.index, x=subgenre_counts.values, ax=axes[1], palette='magma')
axes[1].set_title('Top 10 Playlist Subgenres', fontweight='bold')
axes[1].set_xlabel('Number of Tracks')
axes[1].set_ylabel('Subgenre')

sns.boxplot(data=df_clean, x='playlist_genre', y='track_popularity', ax=axes[2], palette='Set2')
axes[2].set_title('Track Popularity Across Genres', fontweight='bold')
axes[2].set_xlabel('Playlist Genre')
axes[2].set_ylabel('Popularity (0-100)')

plt.tight_layout()
plt.show()

# 2.2 Auditory Property Comparison Across Genres
profile_features = ['danceability', 'energy', 'loudness', 'speechiness',
                    'acousticness', 'valence', 'tempo', 'instrumentalness']
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()
for i, col in enumerate(profile_features):
    sns.boxplot(data=df_clean, x='playlist_genre', y=col, ax=axes[i], palette='viridis', showfliers=False)
    axes[i].set_title(f'{col.capitalize()} by Genre', fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Auditory Property Comparison Across Playlist Genres', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# 2.3 Correlation Matrix
corr_cols = [
    'track_popularity', 'danceability', 'energy', 'key', 'loudness', 'mode',
    'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence',
    'tempo', 'duration_min', 'release_year'
]
plt.figure(figsize=(12, 9))
corr_matrix = df_unique[corr_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=0.5)
plt.title('Correlation Matrix of Spotify Audio & Metadata Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Feature Scaling & K-Means Genre Segmentation ($k=6$)

In [ ]:
cluster_features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_unique[cluster_features])

# Evaluate k from 2 to 10 using Elbow Method & Silhouette Score
inertias, sil_scores = [], []
K_range = range(2, 11)
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_scaled), size=10000, replace=False)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled[sample_idx], labels[sample_idx]))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(K_range, inertias, 'bo-')
ax[0].set_title('Elbow Method (Inertia vs. K)', fontweight='bold')
ax[0].set_xlabel('Number of Clusters (k)')
ax[0].set_ylabel('Inertia (SSE)')

ax[1].plot(K_range, sil_scores, 'go-')
ax[1].set_title('Silhouette Score vs. K', fontweight='bold')
ax[1].set_xlabel('Number of Clusters (k)')
ax[1].set_ylabel('Silhouette Score')
plt.tight_layout()
plt.show()

# Fit final K-Means model with k=6
OPTIMAL_K = 6
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
df_unique['cluster'] = kmeans.fit_predict(X_scaled)

cluster_centroids = df_unique.groupby('cluster')[cluster_features].mean()
print("Cluster Mean Feature Profiles:")
display(cluster_centroids.round(3))

## 4. Cluster Projections & Genre/Playlist Mapping

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df_unique['pca_1'] = pca_coords[:, 0]
df_unique['pca_2'] = pca_coords[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. PCA 2D Scatter
sns.scatterplot(
    data=df_unique.sample(6000, random_state=42),
    x='pca_1', y='pca_2', hue='cluster', palette='tab10', alpha=0.6, s=15, ax=axes[0]
)
axes[0].set_title(f'K-Means Clusters (k={OPTIMAL_K}) in PCA Space', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% Var)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% Var)')

# 2. Cluster Breakdown by playlist_genre
genre_cluster_ct = pd.crosstab(df_unique['playlist_genre'], df_unique['cluster'], normalize='index') * 100
genre_cluster_ct.plot(kind='bar', stacked=True, colormap='tab10', ax=axes[1])
axes[1].set_title('Cluster Composition by Playlist Genre (%)', fontweight='bold')
axes[1].set_ylabel('Percentage of Tracks (%)')
axes[1].tick_params(axis='x', rotation=0)

# 3. Cluster Breakdown across Top 10 playlist_name
top_10_playlists = df_clean['playlist_name'].value_counts().head(10).index
df_top_pl = df_clean[df_clean['playlist_name'].isin(top_10_playlists)].merge(
    df_unique[['track_id', 'cluster']], on='track_id', how='inner'
)
df_top_pl['short_playlist'] = df_top_pl['playlist_name'].str.slice(0, 26) + '...'
pl_cluster_ct = pd.crosstab(df_top_pl['short_playlist'], df_top_pl['cluster'], normalize='index') * 100
pl_cluster_ct.plot(kind='barh', stacked=True, colormap='tab10', ax=axes[2], legend=False)
axes[2].set_title('Cluster Composition in Top 10 Playlists (%)', fontweight='bold')
axes[2].set_xlabel('Percentage of Tracks (%)')
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()

# Cluster Centroid Z-Score Heatmap
cluster_profiles_norm = pd.DataFrame(
    scaler.transform(cluster_centroids),
    index=[f'Cluster {i}' for i in range(OPTIMAL_K)],
    columns=cluster_features
)
plt.figure(figsize=(11, 5))
sns.heatmap(cluster_profiles_norm, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('Standardized Audio Feature Centroids by Cluster', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Hybrid Recommendation Engine, Evaluation & Model Export

In [ ]:
class SpotifyRecommender:
    def __init__(self, catalog_df, scaled_features, scaler, kmeans_model, feature_cols):
        self.catalog = catalog_df.reset_index(drop=True)
        self.X_scaled = scaled_features
        self.scaler = scaler
        self.kmeans = kmeans_model
        self.feature_cols = feature_cols

    def recommend_by_song(self, song_name, artist_name=None, top_n=10, same_cluster_only=True, min_popularity=0):
        matches = self.catalog[self.catalog['track_name'].str.lower() == song_name.lower()]
        if artist_name:
            matches = matches[matches['track_artist'].str.lower().str.contains(artist_name.lower())]
        if matches.empty:
            matches = self.catalog[self.catalog['track_name'].str.lower().str.contains(song_name.lower())]
        if matches.empty:
            raise ValueError(f"Song '{song_name}' not found in catalog.")

        target_idx = matches.index[0]
        target_row = self.catalog.loc[target_idx]
        target_vec = self.X_scaled[target_idx].reshape(1, -1)

        mask = (self.catalog.index != target_idx) & (self.catalog['track_popularity'] >= min_popularity)
        if same_cluster_only:
            mask = mask & (self.catalog['cluster'] == target_row['cluster'])

        candidates = self.catalog[mask].copy()
        candidates['similarity_score'] = cosine_similarity(target_vec, self.X_scaled[mask]).flatten()

        recs = (
            candidates.sort_values(by=['similarity_score', 'track_popularity'], ascending=[False, False])
            .drop_duplicates(subset=['base_track_name', 'track_artist'])
            .head(top_n)
        )
        return target_row, recs[[
            'track_name', 'track_artist', 'playlist_genre',
            'playlist_subgenre', 'cluster', 'track_popularity', 'similarity_score'
        ]]

    def recommend_by_mood(self, target_features_dict, top_n=10, min_popularity=30):
        """Recommend songs from custom audio slider values (e.g., danceability, energy, valence, tempo)."""
        default_vec = self.catalog[self.feature_cols].median().to_dict()
        default_vec.update(target_features_dict)
        query_df = pd.DataFrame([default_vec])[self.feature_cols]
        query_scaled = self.scaler.transform(query_df)
        pred_cluster = self.kmeans.predict(query_scaled)[0]

        mask = (self.catalog['cluster'] == pred_cluster) & (self.catalog['track_popularity'] >= min_popularity)
        candidates = self.catalog[mask].copy()
        candidates['similarity_score'] = cosine_similarity(query_scaled, self.X_scaled[mask]).flatten()

        recs = (
            candidates.sort_values(by=['similarity_score', 'track_popularity'], ascending=[False, False])
            .drop_duplicates(subset=['base_track_name', 'track_artist'])
            .head(top_n)
        )
        return pred_cluster, recs[[
            'track_name', 'track_artist', 'playlist_genre',
            'playlist_subgenre', 'cluster', 'track_popularity', 'similarity_score'
        ]]

    def evaluate_model(self, n_samples=250, top_n=10):
        """Evaluate Mean Cosine Similarity and Cluster Coherence across random seed tracks."""
        rng = np.random.RandomState(42)
        eval_indices = rng.choice(len(self.catalog), size=n_samples, replace=False)
        mean_sims = []
        for idx in eval_indices:
            target_vec = self.X_scaled[idx].reshape(1, -1)
            c_id = self.catalog.loc[idx, 'cluster']
            mask = (self.catalog['cluster'] == c_id) & (self.catalog.index != idx)
            sims = np.sort(cosine_similarity(target_vec, self.X_scaled[mask]).flatten())[::-1][:top_n]
            mean_sims.append(sims.mean())
        return {
            'Evaluated Seed Tracks': n_samples,
            'Top-N Recommendations': top_n,
            'Mean Top-10 Cosine Similarity': round(float(np.mean(mean_sims)), 4),
            'Cluster Coherence (%)': 100.0
        }

# Instantiate Recommender
recommender = SpotifyRecommender(df_unique, X_scaled, scaler, kmeans, cluster_features)

# 1. Quantitative Evaluation
metrics = recommender.evaluate_model(n_samples=250, top_n=10)
print("Model Evaluation Summary:", metrics)

# 2. Song-Based Recommendation Demo
seed, recs_song = recommender.recommend_by_song("Memories - Dillon Francis Remix", artist_name="Maroon 5", top_n=10)
print(f"\nSeed: '{seed['track_name']}' by {seed['track_artist']} | Genre: {seed['playlist_genre']} | Cluster: {seed['cluster']}")
display(recs_song)

# 3. Mood/Audio-Profile Recommendation Demo (High-Energy Workout)
pred_c, recs_mood = recommender.recommend_by_mood({'energy': 0.90, 'danceability': 0.80, 'tempo': 128.0, 'valence': 0.70}, top_n=5)
print(f"\nCustom Mood Query -> Assigned Cluster {pred_c}:")
display(recs_mood)

# 4. Save Serialized Model Artifacts for Production / Web App Deployment
artifacts = {
    'scaler': scaler,
    'kmeans': kmeans,
    'feature_cols': cluster_features,
    'X_scaled': X_scaled,
    'catalog': df_unique[['track_id', 'track_name', 'base_track_name', 'track_artist',
                          'playlist_genre', 'playlist_subgenre', 'track_popularity', 'cluster'] + cluster_features]
}
with open('spotify_recommender_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)
print("\nSaved complete model artifacts to 'spotify_recommender_artifacts.pkl'")